In [2]:
import thinkpython

# Текст шинжлэх ба үүсгэх

Бид Python-ы үндсэн өгөгдлийн бүтцүүд болох жагсаалт, dict төрөл, tuple төрөл болон тэдгээрийг ашигладаг зарим алгоритмтай танилцсан. Энэ бүлэгт эдгээрийг ашиглан текст шинжлэх, Марковын аргаар текст үүсгэхийг судална:

* Текст шинжлэх нь баримт бичгийн үгсийн статистик хамаарлыг, тухайлбал нэг үгийн араас нөгөө үг орох магадлалыг тодорхойлох арга юм.

* Марковын аргаар текст үүсгэх нь эх тексттэй төстэй үг, хэллэг ашиглан шинэ текст зохиох арга юм.

Эдгээр алгоритм нь чатботын гол бүрэлдэхүүн болох том хэлний загварын (LLM) зарим хэсэгтэй төстэй.

Эхлээд номд үг бүр хэдэн удаа гарсныг тоолно. Дараа нь хос үгсийг судалж, үг тус бүрийн араас орж болох үгсийн жагсаалт гаргана. Марковын текст үүсгэгчийн энгийн хувилбарыг хийж үзээд, дасгалд илүү ерөнхий хувилбарыг бүтээх боломжтой болно.

## Давхардалгүй үгс

Текстийн шинжилгээ хийх эхний алхам болгон Роберт Луис Стивенсоны бичсэн *Доктор Жекилл ба ноён Хайдын хачирхалтай хэрэг* номыг уншиж, өвөрмөц үгсийн тоог тоолъё.
Номыг татаж авах зааврыг энэ бүлгийн дэвтэрт оруулсан болно.

Дараах нүд нь Гутенберг төслөөс номыг татаж авдаг.

In [3]:
download('https://www.gutenberg.org/cache/epub/43/pg43.txt');

Төслийн Гутенбергээс авах боломжтой хувилбар нь номын эхэнд байгаа мэдээлэл, төгсгөлд нь лицензийн мэдээллийг агуулдаг.
Бид 8-р бүлгийн `clean_file`-г ашиглан энэ материалыг устгаж, зөвхөн номын текстийг агуулсан "цэвэр" файлыг бичнэ.

In [4]:
def is_special_line(line):
    return line.strip().startswith('*** ')

In [5]:
def clean_file(input_file, output_file):
    reader = open(input_file)
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)
        
    reader.close()
    writer.close()

In [6]:
filename = 'dr_jekyll.txt'

In [7]:
clean_file('pg43.txt', filename)

Бид файлаас мөрүүдийг уншихын тулд `for` давталт, мөрүүдийг үг болгон хуваахын тулд `split` ашиглана.
Дараа нь өвөрмөц үгсийг бүртгэхийн тулд бид үг бүрийг dict төрөлд түлхүүр болгон хадгална.

In [9]:
unique_words = {}
for line in open(filename):
    seq = line.split()
    for word in seq:
        unique_words[word] = 1

len(unique_words)

dict төрлийн урт нь `6000` гэсэн өвөрмөц үгсийн тоо юм.
Гэхдээ бид тэдгээрийг шалгаж үзвэл зарим нь хүчин төгөлдөр бус үгс байгааг харах болно.

Жишээлбэл, `unique_words` дахь хамгийн урт үгсийг харцгаая.
Бид `sorted`-г ашиглан үгсийг эрэмбэлж, `len` функцийг түлхүүр үгийн аргумент болгон дамжуулж, үгсийг уртаар нь эрэмбэлэх боломжтой.

In [88]:
sorted(unique_words, key=len)[-5:]

Зүсмэлийн индекс `[-5:]` нь эрэмбэлэгдсэн жагсаалтын хамгийн урт үгс болох сүүлийн `5` элементүүдийг сонгоно. 

Жагсаалтад хууль ёсны урт үгс, тухайлбал, "хэрэглэл", "шоколадны өнгөтэй" гэх мэт зураастай үгс багтсан болно.
Гэхдээ хамгийн урт "үг"-ийн зарим нь үнэндээ зураасаар тусгаарлагдсан хоёр үг юм.
Мөн бусад үгэнд цэг, анхаарлын тэмдэг, хашилт зэрэг цэг таслал орно.

Тиймээс, цааш явахаасаа өмнө зураас болон бусад цэг таслалыг авч үзье.

## Цэг тэмдэг

Текст дэх үгсийг тодорхойлохын тулд бид хоёр асуудлыг шийдэх хэрэгтэй.

* Нэг мөрөнд зураас гарч ирэхэд бид үүнийг хоосон зайгаар солих хэрэгтэй -- дараа нь `split`-г ашиглах үед үгсийг салгах болно.

* Үгүүдийг хуваасны дараа бид цэг таслалыг арилгахын тулд `strip` ашиглаж болно.

Эхний асуудлыг шийдвэрлэхийн тулд бид мөр авч, зураасыг хоосон зайгаар сольж, мөрийг хувааж, үр дүнгийн жагсаалтыг буцаадаг дараах функцийг ашиглаж болно.

In [11]:
def split_line(line):
    return line.replace('—', ' ').split()

`split_line` нь зураас биш зөвхөн зураасыг орлуулдаг болохыг анхаарна уу.
Бид дараа нь зураасыг авч үзэх болно.

Энд нэг жишээ байна.

In [12]:
split_line('coolness—frightened')

Одоо үг бүрийн эхэн ба төгсгөлийн цэг таслалыг арилгахын тулд бид `strip` ашиглаж болох ч цэг таслал гэж тооцогддог тэмдэгтүүдийн жагсаалт хэрэгтэй.

Python тэмдэгтийн тэмдэгтүүд нь бараг бүх цагаан толгойн үсэг, тоо, тэмдэг, цэг таслал гэх мэтийг илэрхийлэхэд ашигладаг олон улсын стандарт болох Юникод дээр байдаг.
`unicodedata` модуль нь `category` функцээр хангадаг бөгөөд бид ямар тэмдэгтүүд цэг таслал болохыг ялгах боломжтой.
Захидал өгөгдсөн бол тухайн үсэг нь ямар ангилалд багтах тухай мэдээлэл бүхий мөрийг буцаана.

In [13]:
import unicodedata

unicodedata.category('A')

`A` категорийн мөр нь `Lu` -- `L` нь үсэг, `u` нь том үсэг гэсэн үг.

`.` ангилалын мөр нь `Po` -- `P` нь цэг таслал, `o` нь дэд ангилал нь "бусад" гэсэн үг.

In [14]:
unicodedata.category('.')

Бид `P`-ээр эхэлсэн категори бүхий тэмдэгтүүдийг шалгаснаар номын цэг таслалыг олох боломжтой.
Дараах давталт нь dict төрлийн өвөрмөц цэг таслалыг хадгалдаг.

In [15]:
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

Бид цэг таслалуудын жагсаалтыг гаргахын тулд dict төрлийн түлхүүрүүдийг нэг мөр болгон нэгтгэж болно.

In [16]:
punctuation = ''.join(punc_marks)
print(punctuation)

Номын аль тэмдэгт нь цэг таслал болохыг мэддэг болсон тул бид үг авч, цэг таслалыг эх, төгсгөлөөс нь салгаж, жижиг үсэг рүү хөрвүүлэх функц бичиж болно.

In [17]:
def clean_word(word):
    return word.strip(punctuation).lower()

Энд нэг жишээ байна.

In [18]:
clean_word('“Behold!”')

`strip` тэмдэгтүүдийг эх болон төгсгөлөөс хасдаг тул зөвхөн зураастай үгсийг үлдээдэг.

In [19]:
clean_word('pocket-handkerchief')

Одоо энд `split_line` болон `clean_word` ашиглан номон дахь өвөрмөц үгсийг тодорхойлох давталт байна.

In [20]:
unique_words2 = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        unique_words2[word] = 1

len(unique_words2)

Үг гэж юу болох тухай илүү хатуу тодорхойлолтоор 4000 орчим өвөрмөц үг байдаг.
Мөн бид хамгийн урт үгсийн жагсаалтыг цэвэрлэсэн гэдгийг баталж чадна.

In [21]:
sorted(unique_words2, key=len)[-5:]

Одоо үг бүрийг хэдэн удаа хэрэглэж байгааг харцгаая.

## Үгийн давтамж

Дараах давталт нь өвөрмөц үг бүрийн давтамжийг тооцдог.

In [22]:
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

Бид үгийг анх харахад түүний давтамжийг `1` болгон эхлүүлдэг. Хэрэв бид дараа нь ижил үгийг дахин харвал бид түүний давтамжийг нэмэгдүүлдэг.

Аль үг хамгийн их гарч байгааг харахын тулд бид `items`-г ашиглан `word_counter`-аас түлхүүр-утга хосыг авч, хосын хоёр дахь элемент болох давтамжаар эрэмбэлж болно.
Эхлээд бид хоёр дахь элементийг сонгох функцийг тодорхойлно.

In [23]:
def second_element(t):
    return t[1]

Одоо бид `sorted`-г хоёр түлхүүр үгийн аргументтай ашиглаж болно:

* `key=second_element` гэдэг нь тухайн зүйлсийг үгийн давтамжийн дагуу эрэмбэлнэ гэсэн үг.

* `reverse=True` нь эдгээр зүйлсийг урвуу дарааллаар эрэмбэлэх бөгөөд хамгийн их давтамжтай үгсийг түрүүлж эрэмбэлнэ гэсэн үг.

In [24]:
items = sorted(word_counter.items(), key=second_element, reverse=True)

Хамгийн түгээмэл таван үгийг энд оруулав.

In [25]:
for word, freq in items[:5]:
    print(freq, word, sep='\t')

Дараагийн хэсэгт бид энэ давталтыг функцэд багтаах болно.
Мөн бид үүнийг шинэ функц болох нэмэлт параметрүүдийг үзүүлэхэд ашиглах болно.

## Сонголтот параметрүүд

Бид нэмэлт параметрүүдийг авдаг суулгагдсан функцүүдийг ашигласан.
Жишээлбэл, `round` нь `ndigits` нэртэй нэмэлт параметрүүдийг авдаг бөгөөд энэ нь хэдэн аравтын бутархайг хадгалахыг заадаг.

In [26]:
round(3.141592653589793, ndigits=3)

Гэхдээ энэ нь зөвхөн суулгасан функцүүд биш бөгөөд бид нэмэлт параметрүүдтэй функцүүдийг бичиж болно.
Жишээлбэл, дараах функц нь `word_counter` болон `num` гэсэн хоёр параметрийг авдаг.

In [27]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

Хоёрдахь параметр нь даалгаврын заавар шиг харагдах боловч энэ нь биш - энэ нь нэмэлт параметр юм.

Хэрэв та энэ функцийг нэг аргументаар дуудвал `num` нь `5` болох **үндсэн утгыг** авна.

In [28]:
print_most_common(word_counter)

Хэрэв та энэ функцийг хоёр аргументаар дуудвал хоёр дахь аргументыг анхдагч утгын оронд `num`-д онооно.

In [29]:
print_most_common(word_counter, 3)

Энэ тохиолдолд бид нэмэлт аргументыг анхдагч утгыг **дараана** гэж хэлэх болно.

Хэрэв функц нь шаардлагатай болон нэмэлт параметрүүдтэй бол шаардлагатай бүх параметрүүдийг эхлээд, дараа нь нэмэлт параметрүүдийг оруулах ёстой.

In [30]:
%%expect SyntaxError

def bad_function(n=5, word_counter):
    return None

(section_dictionary_subtraction)=
## dict төрлүүдийн ялгавар

Бид номны зөв бичгийн алдаа шалгахыг хүсч байна гэж бодъё, өөрөөр хэлбэл алдаатай бичсэн үгсийн жагсаалтыг олоорой.
Үүнийг хийх нэг арга бол номноос хүчинтэй үгсийн жагсаалтад байхгүй үгсийг олох явдал юм.
Өмнөх бүлгүүдэд бид Scrabble гэх мэт үгийн тоглоомуудад хүчинтэй гэж үзсэн үгсийн жагсаалтыг ашигласан.
Одоо бид энэ жагсаалтыг ашиглан Роберт Луис Стивенсоны алдааг шалгах болно.

Бид энэ бодлогыг олонлог хасах үйлдэл гэж ойлгож болно, өөрөөр хэлбэл бид нэг багцаас (ном дахь үгс) нөгөө багцад байхгүй бүх үгсийг (жагсаалт дахь үгс) олохыг хүсч байна.

Дараах нүд нь үгийн жагсаалтыг татаж авдаг.

In [31]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Бид өмнө нь хийж байсанчлан `words.txt`-н агуулгыг уншиж, мөрийн жагсаалт болгон хувааж болно.

In [32]:
word_list = open('words.txt').read().split()

Бид үгсийг dict төрөлд түлхүүр болгон хадгалах бөгөөд ингэснээр `in` оператор ашиглан үг зөв эсэхийг хурдан шалгах боломжтой болно.

In [33]:
valid_words = {}
for word in word_list:
    valid_words[word] = 1

Одоо номонд орсон боловч үгийн жагсаалтад ороогүй үгсийг тодорхойлохын тулд бид `subtract`-г ашиглах бөгөөд энэ нь хоёр dict төрлийг параметр болгон авч, нөгөөд байхгүй бүх түлхүүрийг агуулсан шинэ dict төрлийг буцаана.

In [34]:
def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res

Бид үүнийг хэрхэн ашиглахыг энд харуулав.

In [35]:
diff = subtract(word_counter, valid_words)

Алдаатай байж болзошгүй үгсийн жишээ авахын тулд бид `diff` дээр хамгийн түгээмэл үгсийг хэвлэж болно.

In [36]:
print_most_common(diff)

Хамгийн түгээмэл "алдаатай" үгс нь ихэвчлэн нэр, цөөн хэдэн ганц үсэгтэй үгс байдаг (Ноён Утерсон бол доктор Жекиллийн найз, хуульч юм).

Хэрэв бид зөвхөн нэг удаа гарч ирэх үгсийг сонговол тэдгээр нь жинхэнэ үсгийн алдаа байх магадлалтай.
Үүнийг бид зүйлсийг давтаж, `1` давтамжтай үгсийн жагсаалтыг гаргаснаар хийж болно.

In [37]:
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

Жагсаалтын сүүлийн хэдэн элементүүд энд байна.

In [38]:
singletons[-5:]

Тэдгээрийн ихэнх нь үгийн жагсаалтад байхгүй хүчинтэй үгс юм.
Гэхдээ `reindue` нь `reinduce`-н үсгийн алдаа бололтой, тиймээс бид ядаж нэг хууль ёсны алдаа оллоо.

## Санамсаргүй тоо

Марковын текст үүсгэх алхам болгон бид `word_counter`-аас санамсаргүй үгсийн дарааллыг сонгох болно.
Гэхдээ эхлээд санамсаргүй байдлын тухай ярья.

Ижил оролттой бол ихэнх компьютерийн программууд нь **детерминистик** байдаг бөгөөд энэ нь тухайн үед ижил гаралтыг үүсгэдэг гэсэн үг юм.
Детерминизм нь ихэвчлэн сайн зүйл байдаг, учир нь бид ижил тооцоолол ижил үр дүнд хүрнэ гэж найдаж байна.
Зарим програмуудын хувьд бид компьютерийг урьдчилан таамаглах боломжгүй байхыг хүсдэг.
Тоглоом бол нэг жишээ, гэхдээ илүү олон зүйл бий.

Хөтөлбөрийг үнэхээр тодорхой бус болгох нь хэцүү байх болно, гэхдээ үүнийг хуурамчаар хийх арга замууд байдаг.
Нэг нь **псевдор санамсаргүй** тоо үүсгэдэг алгоритмуудыг ашиглах явдал юм.
Псевдор санамсаргүй тоонууд нь үнэхээр санамсаргүй тоо биш, учир нь тэдгээр нь детерминист тооцоогоор үүсгэгддэг, гэхдээ зөвхөн тоонуудыг хараад л санамсаргүй тооноос ялгах боломжгүй юм.

`random` модуль нь псевдор санамсаргүй тоо үүсгэдэг функцээр хангадаг бөгөөд би үүнийг эндээс "санамсаргүй" гэж нэрлэх болно.
Ингэж оруулж ирж болно.

In [39]:
import random

In [40]:
# this cell initializes the random number generator so it 
# generates the same sequence each time the notebook runs.

random.seed(4)

`random` модуль нь `choice` хэмээх функцийг хангадаг бөгөөд жагсаалтаас санамсаргүй байдлаар элемент сонгох ба элемент бүрийг сонгох магадлал ижил байна.

In [41]:
t = [1, 2, 3]
random.choice(t)

Хэрэв та функцийг дахин дуудвал ижил элемент эсвэл өөр элемент гарч ирж магадгүй юм.

In [42]:
random.choice(t)

Урт хугацаанд бид элемент бүрийг ижил тооны удаа авна гэж найдаж байна.

Хэрэв та dict төрөлтэй `choice`-г ашиглавал `KeyError` авна.

In [43]:
%%expect KeyError

random.choice(word_counter)

Санамсаргүй түлхүүр сонгохын тулд та түлхүүрүүдийг жагсаалтад оруулаад `choice` руу залгах хэрэгтэй.

In [44]:
words = list(word_counter)
random.choice(words)

Хэрэв бид үгийн санамсаргүй дарааллыг үүсгэдэг бол энэ нь тийм ч их утгагүй болно.

In [45]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

Асуудлын нэг хэсэг нь зарим үгс бусдаас илүү түгээмэл байдгийг анхаарч үзэхгүй байгаа явдал юм.
Хэрэв бид өөр өөр "жин"-тэй үгсийг сонговол үр дүн нь илүү дээр байх болно, ингэснээр зарим нь бусдаас илүү олон удаа сонгогдоно.

Хэрэв бид `word_counter`-ийн утгыг жин болгон ашиглавал үг бүрийг давтамжаас нь хамаарах магадлалаар сонгоно.

In [46]:
weights = word_counter.values()

`random` модуль нь жинг нэмэлт аргумент болгон авдаг `choices` нэртэй өөр функцээр хангадаг.

In [47]:
random.choices(words, weights=weights)

Мөн сонгох үгийн тоог зааж өгөх өөр нэмэлт аргумент болох `k` шаардлагатай.

In [48]:
random_words = random.choices(words, weights=weights, k=6)
random_words

Үр дүн нь өгүүлбэртэй илүү төстэй зүйлд нэгдэж болох мөрүүдийн жагсаалт юм.

In [49]:
' '.join(random_words)

Хэрэв та номноос санамсаргүй байдлаар үг сонговол үгсийн санг ойлгох болно, гэхдээ дараалсан үгсийн хооронд ямар ч хамаарал байхгүй тул санамсаргүй үгсийн цуваа утга учиртай байх нь ховор.
Жишээлбэл, бодит өгүүлбэрт та "the" гэх мэт өгүүллийн араас нь үйл үг, үйл үг биш байх магадлалтай.
Тиймээс дараагийн алхам бол эдгээр үгсийн хоорондын хамаарлыг харах явдал юм.

## Биграм

Одоо бид нэг үгээр харахын оронд **биграм** гэж нэрлэгддэг хоёр үгийн дарааллыг авч үзэх болно.
Гурван үгийн дарааллыг **триграм**, тодорхойгүй хэдэн үгтэй дарааллыг **n-грам** гэнэ.

Номон дээрх бүх биграммууд болон тус бүр хэдэн удаа гарч ирэхийг олох програм бичье.
Үр дүнг хадгалахын тулд бид хаана гэсэн dict төрлийг ашиглах болно

* Түлхүүрүүд нь биграммыг төлөөлөх тэмдэгт мөрүүд ба 

* Утгууд нь давтамжийг илэрхийлэх бүхэл тоонууд юм.

Үүнийг `bigram_counter` гэж нэрлэе.

In [50]:
bigram_counter = {}

Дараах функц нь хоёр мөрийн жагсаалтыг параметр болгон авдаг.
Эхлээд энэ нь dict төрөлд түлхүүр болгон ашиглаж болох хоёр мөрийг бий болгодог.
Дараа нь энэ түлхүүр байхгүй бол `bigram_counter`-д нэмдэг, хэрэв байгаа бол давтамжийг нэмэгдүүлнэ.

In [51]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

Номыг уншиж байхдаа бид дараалсан хос үг бүрийг хянаж байх ёстой.
Тиймээс хэрэв бид "хүн үнэхээр нэг биш" гэсэн дарааллыг харвал "хүн бол", "биш", "үнэхээр биш" гэх мэт биграммуудыг нэмнэ.

Эдгээр биграммуудыг бүртгэхийн тулд бид `window` нэртэй жагсаалтыг ашиглах болно, учир нь энэ нь номын хуудсууд дээгүүр гулсдаг цонхтой адил бөгөөд нэг дор хоёр үг харуулдаг.
Эхлээд `window` хоосон байна.

In [52]:
window = []

Бид үгсийг нэг нэгээр нь боловсруулахын тулд дараах функцийг ашиглана.

In [53]:
def process_word(word):
    window.append(word)
    
    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

Энэ функцийг анх удаа дуудахдаа өгөгдсөн үгийг `window`-д хавсаргана.
Цонхонд ганц үг байгаа тул биграмм хараахан байхгүй байгаа тул функц дуусна.

Хоёр дахь удаагаа дуудагдах бүртээ `window`-д хоёр дахь үг нэмнэ.
Цонхонд хоёр үг байгаа тул биграм бүр хэдэн удаа гарч байгааг хянахын тулд `count_bigram` руу залгадаг.
Дараа нь цонхноос эхний үгийг арилгахын тулд `pop` ашигладаг.

Дараах программ нь номон дээрх үгсийг нэг нэгээр нь эргүүлж боловсруулдаг.

In [54]:
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

Үүний үр дүнд биграмм бүрээс хэдэн удаа гарч ирэхийг харуулсан dict төрөл гарч ирнэ.
Бид `print_most_common`-г ашиглан хамгийн түгээмэл биграммуудыг харах боломжтой.

In [55]:
print_most_common(bigram_counter)

Эдгээр үр дүнг харвал бид аль хос үгс хамтдаа гарч ирэх магадлал өндөр байгааг олж мэдэх боломжтой.
Бид үүнтэй адил санамсаргүй текст үүсгэхийн тулд үр дүнг ашиглаж болно.

In [56]:
random.seed(0)

In [57]:
bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

`bigrams` is a list of the bigrams that appear in the books.
`weights` нь тэдгээрийн давтамжийн жагсаалт тул `random_bigrams` нь биграммыг сонгох магадлал нь түүний давтамжтай пропорциональ байх жишээ юм. 

Үр дүн нь энд байна.

In [58]:
for pair in random_bigrams:
    print(' '.join(pair), end=' ')

Текст үүсгэх ийм арга нь санамсаргүй үгсийг сонгохоос хамаагүй дээр боловч тийм ч их утгагүй хэвээр байна.

## Марковын шинжилгээ

We can do better with Markov chain text analysis, which computes, for each word in a text, the list of words that come next.
Жишээ болгон бид Monty Python дууны *Eric, the Half a Bee* дууны эдгээр үгэнд дүн шинжилгээ хийх болно:

In [59]:
song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

Үр дүнг хадгалахын тулд бид үг бүрээс түүнийг дагасан үгсийн жагсаалтад буулгах dict төрлийг ашиглана.

In [60]:
successor_map = {}

Жишээ болгон дууны эхний хоёр үгнээс эхэлье.

In [61]:
first = 'half'
second = 'a'

Хэрэв эхний үг нь `successor_map`-д байхгүй бол бид эхний үгнээс хоёр дахь үгийг агуулсан жагсаалтад буулгах шинэ зүйлийг нэмэх хэрэгтэй.

In [62]:
successor_map[first] = [second]
successor_map

Хэрэв эхний үг аль хэдийн dict төрөлд байгаа бол бид үүнийг хайж олоод өнөөг хүртэл үзсэн залгамжлагчдын жагсаалтыг гаргаж, шинийг хавсаргаж болно.

In [63]:
first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

Дараах функц нь эдгээр алхмуудыг багтаасан болно.

In [64]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Хэрэв ижил биграмм нэгээс олон удаа гарч ирвэл хоёр дахь үгийг жагсаалтад нэгээс олон удаа нэмнэ.
Ийнхүү `successor_map` залгамжлагч бүр хэдэн удаа гарч ирснийг бүртгэдэг.

Өмнөх хэсэгт бичсэнчлэн бид `window` нэртэй жагсаалтыг дараалсан хос үгсийг хадгалахад ашиглах болно.
Мөн бид үгсийг нэг нэгээр нь боловсруулахын тулд дараах функцийг ашиглана.

In [65]:
def process_word_bigram(word):
    window.append(word)
    
    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

Дууны үгийг боловсруулахдаа үүнийг хэрхэн ашигладаг талаар эндээс үзнэ үү.

In [66]:
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

Ингээд үр дүн нь энд байна.

In [67]:
successor_map

`half` гэдэг үгийн араас `a`, `not`, `the` гэсэн үг орж болно.
`a` гэдэг үгийн араас `bee` эсвэл `vis` орж болно.
Бусад үгсийн ихэнх нь зөвхөн нэг удаа гарч ирдэг тул тэдний араас ганц үг ордог.

Одоо номонд дүн шинжилгээ хийцгээе.

In [68]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

Бид ямар ч үгийг хайж, түүнийг дагаж болох үгсийг олох боломжтой.

In [69]:
# I used this cell to find a predecessor with a good number of possible successors
# and at least one repeated word.

def has_duplicates(t):
    return len(set(t)) < len(t)

for key, value in successor_map.items():
    if len(value) == 7 and has_duplicates(value):
        print(key, value)

In [70]:
successor_map['going']

Энэхүү залгамжлагчдын жагсаалтад `to` гэдэг үг гурван удаа, бусад залгамжлагчид нэг л удаа гарч ирснийг анхаарна уу.

## Текст үүсгэх

Бид өмнөх хэсгийн үр дүнг ашиглан эх хувилбартай адил дараалсан үгсийн хооронд шинэ текст үүсгэх боломжтой.
Энэ нь дараах байдалтай байна.

* Текстэнд гарч буй аливаа үгнээс эхлэн бид түүний залгамжлагчийг хайж, санамсаргүй байдлаар нэгийг нь сонгоно.

* Дараа нь бид сонгосон үгийг ашиглан түүний залгамжлагчдыг хайж, санамсаргүй байдлаар нэгийг нь сонгоно.

Бид хүссэнээрээ олон үг үүсгэхийн тулд энэ үйл явцыг давтаж болно.
Жишээ болгож `although` гэдэг үгээр эхэлье.
Үүнийг дагаж болох үгс энд байна.

In [71]:
word = 'although'
successors = successor_map[word]
successors

In [72]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(2)

Бид ижил магадлалтай жагсаалтаас сонгохдоо `choice` ашиглаж болно.

In [73]:
word = random.choice(successors)
word

Жагсаалтад нэг үг нэгээс олон удаа гарч ирвэл сонгогдох магадлал өндөр.

Эдгээр алхмуудыг давтаж, бид илүү урт цуврал үүсгэхийн тулд дараах давталтыг ашиглаж болно.

In [74]:
for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

Үр дүн нь жинхэнэ өгүүлбэр шиг сонсогддог ч нэг их утгагүй хэвээр байна.

Бид `successor_map` дээр нэгээс олон үгийг түлхүүр болгон ашиглах нь илүү дээр байх болно.
Жишээлбэл, бид биграмм эсвэл триграм бүрээс дараагийн үгсийн жагсаалт руу буулгах dict төрөл хийж болно.
Дасгалын хувьд та энэ шинжилгээг хэрэгжүүлж, үр дүн нь ямар байхыг харах боломжтой болно.

(section_debugging_12)=
## Алдаа олж засварлах

Энэ үед бид илүү их хэмжээний программ бичиж байгаа бөгөөд та алдаа олж засварлахэд илүү их цаг зарцуулж байгааг анзаарч магадгүй.
Хэрэв та хэцүү алдаан дээр гацсан бол дараах хэдэн зүйлийг туршиж үзэх хэрэгтэй:

* Унших: Кодоо шалгаж, өөртөө дахин уншаад, таны хэлэх гэж буй зүйл байгаа эсэхийг шалгаарай.

* Ажиллаж байна: Өөрчлөлт хийж, өөр хувилбаруудыг ажиллуулж туршина. Хэрэв та зөв зүйлийг програмын зөв газарт байрлуулбал асуудал тодорхой болно, гэхдээ заримдаа та шатыг барих хэрэгтэй болдог.

* Бодох: Бодох цаг гарга! Энэ нь ямар төрлийн алдаа вэ: синтакс, ажиллах хугацаа,
    эсвэл семантик уу? Алдааны мэдэгдлээс ямар мэдээлэл авч болох вэ?
    эсвэл хөтөлбөрийн гаралтаас үү? Ямар төрлийн алдаа гарч болох вэ
    Таны харж байгаа асуудал? Та хамгийн сүүлд юуг өөрчилсөн бэ?
    асуудал гарсан уу?

* Rubberducking: Хэрэв та хэн нэгэнд асуудлаа тайлбарлавал заримдаа та үүнийг олж мэднэ
    асуултаа асууж дуусгахаасаа өмнө хариул. Ихэнхдээ танд хэрэггүй
    нөгөө хүн; чи зүгээр л резинэн нугастай ярьж болно. Тэгээд тэр
    **резин нугас хэмээх алдартай стратегийн гарал үүсэл
    алдаа олж засварлах**. Би үүнийг зохиогүй -- харна уу
    <https://en.wikipedia.org/wiki/Rubber_duck_debugging>.

* Ухрах: Хэзээ нэгэн цагт хамгийн сайн хийх зүйл бол нөөцлөх явдал юм-- саяхныг буцаах
    өөрчлөлтүүд -- ажиллаж байгаа програм руу орох хүртэл. Дараа нь та сэргээн босгох ажлыг эхлүүлж болно.
    
* Амрах: Хэрэв та тархидаа завсарлага өгвөл тэр хэзээ нэгэн цагт таны асуудлыг олох болно.

Эхлэн програмистууд заримдаа эдгээр үйлдлүүдийн аль нэгэнд гацаж, бусдыг нь мартдаг. Үйл ажиллагаа бүр өөрийн гэсэн бүтэлгүйтлийн горимтой ирдэг.

Жишээлбэл, хэрэв асуудал нь бичгийн зүйн алдаатай бол кодыг унших нь ажиллах болно, гэхдээ асуудал нь үзэл баримтлалын үл ойлголцол юм.
Хэрэв та програмаа юу хийдгийг ойлгохгүй байвал 100 удаа уншаад алдааг хэзээ ч харахгүй болно, учир нь алдаа таны толгойд байгаа.

Хэрэв та жижиг, энгийн туршилтуудыг ажиллуулж байгаа бол туршилтыг ажиллуулах нь үр дүнтэй байх болно.
Гэхдээ хэрэв та өөрийн кодыг бодолгүйгээр эсвэл уншихгүйгээр туршилт хийвэл юу болж байгааг ойлгоход удаан хугацаа шаардагдана.

Та бодох цаг гаргах хэрэгтэй. Алдаа олж засварлах нь туршилтын шинжлэх ухаантай адил юм. Асуудал юу болох талаар та дор хаяж нэг таамаглалтай байх ёстой. Хэрэв хоёр ба түүнээс дээш боломж байгаа бол тэдгээрийн аль нэгийг нь арилгах тестийг бодож үзээрэй.

Гэхдээ хамгийн сайн алдаа олж засварлах аргууд ч гэсэн хэтэрхий олон байвал бүтэлгүйтэх болно
алдаа, эсвэл таны засах гэж буй код хэтэрхий том бөгөөд төвөгтэй байвал.
Заримдаа хамгийн сайн сонголт бол ухрах, хөтөлбөрийг хүртэл хялбаршуулах явдал юм
Та ажиллаж байгаа зүйлдээ буцаж очно.

Анхлан програмистууд чадахгүй учраас ухрах дургүй байдаг
кодын мөрийг устгахын тулд зогсох (буруу байсан ч гэсэн). Хэрэв энэ нь таныг болгодог бол
Хэрэв та илүү сайн байгаа бол эхлэхээсээ өмнө програмаа өөр файл руу хуулна уу
үүнийг буулгах. Дараа нь та хэсгүүдийг нэг нэгээр нь хуулж болно.

Хэцүү алдааг олохын тулд унших, гүйх, эргэцүүлэх, ухрах, заримдаа амрах шаардлагатай байдаг.
Хэрэв та эдгээр үйл ажиллагааны аль нэгэнд гацсан бол бусдыг нь туршиж үзээрэй.

## Нэр томьёоны тайлбар

**анхдагч утга:**
Аргумент өгөөгүй үед параметрт олгодог утга.

**анхдагч утгыг солих:**
Аргумент дамжуулж, анхдагч утгыг орлуулах.

**детерминист:**
Ижил оролт өгвөл ажиллуулах бүрд ижил үйлдэл хийдэг програмыг детерминист гэнэ.

**псевдо-санамсаргүй:**
Санамсаргүй мэт харагдах боловч детерминист програмаар үүсгэсэн тоонуудын дараалал.

**биграм:**
Ихэвчлэн үгсээс бүрдэх, хоёр элементийн дараалал.

**триграм:**
Гурван элементийн дараалал.

**n-грам:**
Элементийн тоог нь тодорхой заагаагүй дараалал.


## Дасгалууд

### Виртуал туслахаас асуух

`add_bigram`-д `if` заавар нь шинэ жагсаалт үүсгэх эсвэл тухайн түлхүүр нь dict төрөлд байгаа эсэхээс хамаарч байгаа жагсаалтад элемент нэмнэ.

In [75]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

dict төрлүүд нь `setdefault` хэмээх аргыг өгдөг бөгөөд бид ижил зүйлийг илүү товчоор хийх боломжтой.
Энэ нь хэрхэн ажилладаг талаар виртуал туслахаас асуу эсвэл `add_word`-г виртуал туслах руу хуулаад "Та үүнийг `setdefault` ашиглан дахин бичиж чадах уу?"

Энэ бүлэгт бид Марковын гинжин текстийн дүн шинжилгээ хийх, үүсгэх ажлыг хэрэгжүүлсэн.
Хэрэв та сонирхож байгаа бол энэ сэдвээр нэмэлт мэдээллийг виртуал туслахаас асууж болно.
Таны мэдэж болох зүйлсийн нэг бол виртуал туслахууд нь олон талаараа ижил төстэй алгоритмуудыг ашигладаг, гэхдээ бас чухал арга замаар ялгаатай байдаг.
VA-аас "GPT болон Марковын гинжин текстийн шинжилгээ зэрэг том хэлний загваруудын хооронд ямар ялгаа байдаг вэ?"

### Дасгал

Триграм (гурван үгийн дараалал) бүр хэдэн удаа гарч ирэхийг тоолох функц бич. 
Хэрэв та өөрийн функцийг _Dr. Жекилл, ноён Хайд_, та хамгийн түгээмэл триграм бол "хуульч хэлэв" гэдгийг олж мэдэх хэрэгтэй.

Зөвлөгөө: `count_trigram` нэртэй `count_bigram`-тэй төстэй функц бичнэ үү. Дараа нь `process_word_bigram`-тэй төстэй `process_word_trigram` нэртэй функц бич.

In [76]:
# Solution goes here

In [77]:
# Solution goes here

Та номыг уншиж, үгсийг боловсруулахдаа дараах давталтыг ашиглаж болно.

In [78]:
trigram_counter = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Дараа нь `print_most_common` ашиглан номон дээрх хамгийн түгээмэл триграммуудыг олоорой.

In [79]:
print_most_common(trigram_counter)

### Дасгал

Одоо биграм бүрээс боломжит залгамжлагчдын жагсаалт хүртэлх зураглал бүхий Марковын гинжин текстийн шинжилгээг хэрэгжүүлье.

`add_bigram`-аас эхлэн `add_trigram` нэртэй функцийг бичээд гурван үгийн жагсаалтыг аваад `successor_map`-д нэг зүйлийг нэмж эсвэл шинэчилж, эхний хоёр үгийг түлхүүр, гурав дахь үгийг залгамжлагч болгон ашиглана.

In [80]:
# Solution goes here

`process_word_trigram`-ийн `add_trigram` гэж нэрлэдэг хувилбар энд байна.

In [81]:
def process_word_trigram(word):
    window.append(word)
    
    if len(window) == 3:
        add_trigram(window)
        window.pop(0)

Та "Eric, the Half a Bee" дууны үгээр үйл ажиллагаагаа шалгахын тулд дараах давталтыг ашиглаж болно.

In [82]:
successor_map = {}
window = []

for string in song.split():
    word = string.strip(punctuation).lower()
    process_word_trigram(word)

Хэрэв таны функц төлөвлөсний дагуу ажиллаж байвал өмнөх `('half', 'a')` нь `'bee'` дан элемент бүхий жагсаалтад буулгах ёстой.
Үнэн хэрэгтээ энэ дууны биграм бүр нэг л удаа гарч ирдэг тул `successor_map` дахь бүх утгууд нэг элементтэй байдаг.

In [83]:
successor_map

Та ном дээрх үгсээр үйл ажиллагаагаа шалгахын тулд дараах давталтыг ашиглаж болно.

In [84]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Дараагийн дасгал дээр та шинэ санамсаргүй текст үүсгэхийн тулд үр дүнг ашиглах болно.

### Дасгал

Энэ дасгалын хувьд бид `successor_map` нь биграмм бүрээс түүнийг дагасан үгсийн жагсаалт хүртэлх dict төрөл гэж таамаглах болно.

In [85]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(3)

Санамсаргүй текст үүсгэхийн тулд бид `successor_map`-аас санамсаргүй түлхүүр сонгох замаар эхэлнэ.

In [86]:
successors = list(successor_map)
bigram = random.choice(successors)
bigram

Одоо эдгээр алхмуудыг дагаж `50` илүү олон үг үүсгэх давталт бичнэ үү:

1. `successor_map` дотроос `bigram`-г дагаж болох үгсийн жагсаалтыг хай.

2. Тэдгээрийн аль нэгийг нь санамсаргүй байдлаар сонгоод хэвлэ.

3. Дараагийн давталтын хувьд `bigram`-ийн хоёр дахь үг болон сонгосон залгамжлагчийг агуулсан шинэ биграмм хийнэ.

Жишээлбэл, бид `('doubted', 'if')` биграммыг эхлүүлж, түүний залгамжлагчаар `'from'`-г сонговол дараагийн биграм нь `('if', 'from')` болно.

In [89]:
# Solution goes here

Хэрэв бүх зүйл ажиллаж байгаа бол үүсгэсэн текст нь хэв маягийн хувьд анхныхтай ижил төстэй бөгөөд зарим хэллэг нь утга учиртай боловч текст нэг сэдвээс нөгөөд тэнүүчилж магадгүй гэдгийг олж мэдэх хэрэгтэй.

Урамшууллын дасгалын хувьд `successor_map`-д триграммуудыг түлхүүр болгон ашиглахын тулд сүүлийн хоёр дасгал руу шийдлээ өөрчилж, үр дүнд нь ямар нөлөө үзүүлэхийг хараарай.